## 📦 Import Libraries
Import required libraries:
- **GEOparse** — fetch metadata and file links from NCBI GEO
- **os** — handle file paths and directory creation
- **pandas** — store and manipulate sample metadata as a table

In [1]:
import GEOparse
import os
import pandas as pd

## ⚙️ Configuration
Define the GEO accession ID for the dataset and set the local directory 
where raw data will be stored. The directory is created automatically 
if it does not already exist.

In [2]:
GEO_ID = "GSE279086"
RAW_DIR = "../data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

## 🌐 Fetch GEO Series Metadata
Connect to NCBI GEO and download the soft file for GSE279086.
This retrieves all sample-level metadata including GSM accession IDs,
sample titles, tissue type, cell type, and disease condition for all 40 samples.

In [ ]:
gse = GEOparse.get_GEO(GEO_ID, destdir=RAW_DIR, silent=False)

## 📋 Parse Sample Metadata
Extract biological characteristics for each sample from the GEO metadata.
Each sample has multiple characteristics stored as key:value pairs
(tissue, cell type, disease condition). These are parsed into a clean
DataFrame with one row per sample, indexed from 1 to 40.

In [ ]:
samples = []
for gsm_name, gsm in gse.gsms.items():
    title = gsm.metadata.get('title', [''])[0]
    chars = gsm.metadata.get('characteristics_ch1', [])
    
    # Extract all characteristics into a dict
    char_dict = {}
    for c in chars:
        if ':' in c:
            key, val = c.split(':', 1)
            char_dict[key.strip()] = val.strip()
    
    samples.append({
        'GSM': gsm_name,
        'Title': title,
        **char_dict
    })

df_samples = pd.DataFrame(samples)

# Start index from 1
df_samples.index = range(1, len(df_samples) + 1)
df_samples.index.name = "Sample"

print(f"Total samples: {len(df_samples)}")
print(f"\nColumns found: {df_samples.columns.tolist()}")
df_samples

## 🔢 Disease Distribution
Summarize the number of samples per disease condition.
This confirms our experimental design:
- **Type 1 Diabetes** — disease group
- **Lean Control** — healthy control group

In [7]:
print("Disease distribution:")
print(df_samples['disease'].value_counts())

Disease distribution:
disease
Type 1 Diabetes    28
Lean Control       12
Name: count, dtype: int64


## 💾 Save Sample Manifest
Export the complete sample metadata table to a CSV file.
This manifest serves as the master reference throughout the pipeline,
mapping GSM accession IDs to sample titles and disease conditions
for all downstream analysis scripts.

In [8]:
df_samples.to_csv("../output/sample_manifest.csv", index=True)
print("✅ Saved: output/sample_manifest.csv")

✅ Saved: output/sample_manifest.csv


## 🔍 Explore GEO Supplementary Files
List the FTP URLs of all supplementary files available for each sample.
GEO hosts multiple file types per sample — this step identifies
which files are available before we decide what to download.

In [ ]:
import subprocess

# Create download directory
DOWNLOAD_DIR = "../data/raw"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Get FTP links for each sample
for idx, row in df_samples.iterrows():
    gsm_name = row['GSM']
    gsm = gse.gsms[gsm_name]
    
    # Get supplementary files
    suppl_files = gsm.metadata.get('supplementary_file_1', [])
    
    print(f"\nSample {idx}: {gsm_name}")
    for f in suppl_files:
        print(f"  → {f}")

## 🗂️ Inspect File Structure for One Sample
Examine all supplementary file keys for a single representative sample (GSM8561110).
Each sample on GEO contains 6 files — raw and processed versions of the
three standard 10X Genomics files:
- **barcodes** (supplementary_file_1) — cell barcodes
- **features** (supplementary_file_3) — gene identifiers  
- **matrix** (supplementary_file_5) — UMI count matrix

We download only the **raw files** (files 1, 3, 5) to ensure we start
from unprocessed counts for a fully reproducible pipeline.

In [ ]:

gsm_test = gse.gsms['GSM8561110']

print("All metadata keys:")
for key, val in gsm_test.metadata.items():
    if 'suppl' in key.lower():
        print(f"\n{key}:")
        for v in val:
            print(f"  {v}")

## ⬇️ Download Raw 10X Files for All 40 Samples
Download the three raw 10X Genomics files for each sample from the NCBI GEO FTP server:
- **barcodes.tsv.gz** — unique cell barcode sequences
- **features.tsv.gz** — Ensembl gene IDs and gene symbols
- **matrix.mtx.gz** — sparse UMI count matrix (cells × genes)

Files are renamed to the standard 10X format required by scanpy's `read_10x_mtx()`.
Already downloaded files are automatically skipped, making this cell safe to re-run
in case of connection interruptions.

In [ ]:
import urllib.request

# We only want raw files: barcodes(1), features(3), matrix(5)
RAW_KEYS = ['supplementary_file_1', 'supplementary_file_3', 'supplementary_file_5']

for idx, row in df_samples.iterrows():
    gsm_name = row['GSM']
    sample_title = row['Title']
    gsm = gse.gsms[gsm_name]

    # Create sample folder
    sample_dir = os.path.join(DOWNLOAD_DIR, sample_title)
    os.makedirs(sample_dir, exist_ok=True)

    print(f"\n▶ Sample {idx}/40: {gsm_name} — {sample_title}")

    for key in RAW_KEYS:
        url = gsm.metadata.get(key, [''])[0]
        if not url:
            print(f"  ⚠️ {key} not found")
            continue

        filename = url.split('/')[-1]
        
        # Rename to standard 10X names
        if 'barcodes' in filename:
            out_name = 'barcodes.tsv.gz'
        elif 'features' in filename:
            out_name = 'features.tsv.gz'
        elif 'matrix' in filename:
            out_name = 'matrix.mtx.gz'
        
        out_path = os.path.join(sample_dir, out_name)

        if os.path.exists(out_path):
            print(f"  ✅ Already exists: {out_name}")
            continue

        print(f"  ⬇ Downloading: {out_name}")
        urllib.request.urlretrieve(url, out_path)
        print(f"  ✔ Saved: {out_path}")

print("\n🎉 All downloads complete!")

## ✅ Verify Downloads
Confirm that all 40 sample folders contain exactly the 3 required files.
Any sample with missing files is flagged with a warning so it can be
re-downloaded before proceeding to the QC step.
A final success message confirms the dataset is complete and ready
for quality control analysis in Script 02.

In [ ]:

import os

raw_dir = "../data/raw"
sample_dirs = sorted([d for d in os.listdir(raw_dir) if d.startswith("S_")])

print(f"Total sample folders: {len(sample_dirs)}\n")

all_ok = True
for sample in sample_dirs:
    files = os.listdir(os.path.join(raw_dir, sample))
    expected = {'barcodes.tsv.gz', 'features.tsv.gz', 'matrix.mtx.gz'}
    missing = expected - set(files)
    if missing:
        print(f"⚠️  {sample} — missing: {missing}")
        all_ok = False
    else:
        print(f"✅ {sample}")

print(f"\n{'🎉 All 40 samples verified!' if all_ok else '⚠️ Some samples incomplete!'}")

Total sample folders: 40

✅ S_1907_004567
✅ S_1907_004614
✅ S_1907_004802
✅ S_1907_004896
✅ S_1907_005225
✅ S_2006_004078
✅ S_2007_002809
✅ S_2007_002950
✅ S_2007_002997
✅ S_2007_003044
✅ S_2007_003091
✅ S_2007_003138
✅ S_2007_003793
✅ S_2007_003840
✅ S_2007_003934
✅ S_2007_004028
✅ S_2007_004216
✅ S_2007_004263
✅ S_2103_004019
✅ S_2103_004028
✅ S_2103_004037
✅ S_2103_004046
✅ S_2103_004064
✅ S_2103_004073
✅ S_2103_004091
✅ S_2103_004100
✅ S_2103_004109
✅ S_2103_004118
✅ S_2103_004127
✅ S_2103_004145
✅ S_2103_004154
✅ S_2103_004163
✅ S_2103_004181
✅ S_2107_023520
✅ S_2107_023538
✅ S_2107_023547
✅ S_2107_023556
✅ S_2107_023592
✅ S_2107_023610
✅ S_2107_023619

🎉 All 40 samples verified!
